
## Evaluate and Deploy an LLM Application with FastAPI

### Aim

To develop an LLM-powered API using FastAPI and evaluate its responses using a simple test case.

### Algorithm

1. Import FastAPI and the LLM library.
2. Initialize the language model.
3. Create a FastAPI application.
4. Define a request schema.
5. Create an endpoint that accepts a question.
6. Generate an answer using the LLM.
7. Evaluate the response for non-empty content.
8. Run the application using Uvicorn.
9. Test the endpoint.

### Dependies

In [1]:
%pip install -q fastapi uvicorn langchain langchain-google-genai requests

Note: you may need to restart the kernel to use updated packages.


In [1]:
import os
import getpass

os.environ["GOOGLE_API_KEY"] = getpass.getpass(
    "Enter your Gemini API key: "
)

print("Gemini API key configured successfully!")

Gemini API key configured successfully!


### Import Libraries and Initialize FastAPI

In [2]:
from fastapi import FastAPI
from pydantic import BaseModel
from langchain_google_genai import ChatGoogleGenerativeAI

app = FastAPI(title="Generative AI API")

llm = ChatGoogleGenerativeAI(
    model="gemini-3.8-flash",
    temperature=0
)

print("FastAPI application and Gemini model initialized successfully!")

FastAPI application and Gemini model initialized successfully!


### Defining the Request Schema

In [3]:
class Question(BaseModel):
    question: str

print("Request schema defined successfully!")

Request schema defined successfully!


In [4]:
@app.get("/")
def home():
    return {"message": "LLM API is running"}


@app.post("/generate")
def generate(request: Question):
    response = llm.invoke(request.question)
    answer = response.content

    # Convert Gemini content blocks into a normal string if necessary
    if isinstance(answer, list):
        answer = "".join(
            item.get("text", "") if isinstance(item, dict) else str(item)
            for item in answer
        )

    return {
        "question": request.question,
        "answer": answer,
        "evaluation": {
            "non_empty": bool(answer.strip()),
            "character_count": len(answer)
        }
    }

print("FastAPI endpoints defined successfully!")

FastAPI endpoints defined successfully!


### Run the FastAPI Application Using Uvicorn

In [5]:
import threading
import time
import uvicorn

config = uvicorn.Config(
    app,
    host="127.0.0.1",
    port=8000,
    log_level="warning"
)

server = uvicorn.Server(config)

server_thread = threading.Thread(
    target=server.run,
    daemon=True
)

server_thread.start()
time.sleep(2)

print("FastAPI server started successfully!")
print("API URL: http://127.0.0.1:8000")
print("Swagger UI: http://127.0.0.1:8000/docs")

FastAPI server started successfully!
API URL: http://127.0.0.1:8000
Swagger UI: http://127.0.0.1:8000/docs


In [ ]:
import requests

# Test the home endpoint
home_response = requests.get(
    "http://127.0.0.1:8000/"
)

print("Home Endpoint:", home_response.json())

# Test the LLM generation endpoint
response = requests.post(
    "http://127.0.0.1:8000/generate",
    json={"question": "What is Generative AI?"},
    timeout=120
)

print("\nHTTP Status Code:", response.status_code)
print("\nAPI Response:")
print(response.json())

Home Endpoint: {'message': 'LLM API is running'}


In [ ]:
server.should_exit = True
print("FastAPI server stopped successfully!")

### Output

The FastAPI application was started successfully. The `/generate` endpoint accepted a question, generated an answer using Gemini 3.8 Flash, and returned the answer with evaluation details.

### Result

An LLM application was developed and exposed through a FastAPI endpoint. A basic response-quality check verified that the answer was non-empty. This is a functional check, not a complete evaluation of factual accuracy.